# Laboratorio 02: NumPy, pandas, Ridge y validación cruzada

En este laboratorio continuamos con NumPy, pandas y scikit-learn. Verás cuándo una regresión lineal sobreajusta, cómo Ridge lo controla y cómo elegir su hiperparámetro $\alpha$ con validación cruzada usando `GridSearchCV`.

Ridge minimiza el error cuadrático más una penalización sobre el tamaño de los coeficientes:

$$\sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} w_j^2$$

Con $\alpha = 0$ es una regresión lineal normal. Mientras más grande es $\alpha$, más pequeños quedan los coeficientes.

## Preparación del entorno

Igual que en el laboratorio 01. Hacer antes de la clase, porque se necesita internet para instalar los paquetes. El archivo `diabetes.csv` debe estar en la misma carpeta que este notebook.

### Opción 1: uv

```bash
cd labs/02
uv venv
source .venv/bin/activate
uv add numpy pandas matplotlib scikit-learn jupyterlab
jupyter lab
```

### Opción 2: pip

```bash
cd labs/02
python -m venv .venv
source .venv/bin/activate
python -m pip install numpy pandas matplotlib scikit-learn jupyterlab
jupyter lab
```

## 1. NumPy: herramientas nuevas

Funciones de NumPy que no vimos en el laboratorio 01 y que usaremos más adelante.

In [23]:
import numpy as np

# np.arange(inicio, fin, paso): como range(), pero devuelve un arreglo. El fin NO se incluye.
print(np.arange(0, 10, 2))

# np.linspace(inicio, fin, n): n valores espaciados uniformemente. El fin SÍ se incluye.
print(np.linspace(0, 1, 5))

# np.logspace(a, b, n): n valores entre 10**a y 10**b, espaciados en escala logarítmica.
# Lo usaremos para probar valores de alpha que van desde muy pequeños hasta muy grandes.
alphas = np.logspace(-2, 2, 5)
print(alphas)  # [0.01, 0.1, 1, 10, 100]

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[1.e-02 1.e-01 1.e+00 1.e+01 1.e+02]


In [24]:
# Una matriz de 4 filas (observaciones) y 3 columnas (características)
M = np.array([[1.0, 200.0, 3.0],
              [2.0, 180.0, 1.0],
              [3.0, 260.0, 2.0],
              [4.0, 240.0, 6.0]])

# Sin axis, la operación usa todos los elementos
print("Promedio total:", M.mean())

# axis=0 recorre las filas: devuelve un resultado por columna
print("Promedio por columna:", M.mean(axis=0))
print("Desv. estándar por columna:", M.std(axis=0))

# axis=1 recorre las columnas: devuelve un resultado por fila
print("Promedio por fila:", M.mean(axis=1))

Promedio total: 75.16666666666667
Promedio por columna: [  2.5 220.    3. ]
Desv. estándar por columna: [ 1.11803399 31.6227766   1.87082869]
Promedio por fila: [68.         61.         88.33333333 83.33333333]


In [ ]:
# Broadcasting: al operar M (forma (4, 3)) con un arreglo de forma (3,),
# NumPy repite ese arreglo para cada fila de M.
media = M.mean(axis=0)  # forma (3,)
desv = M.std(axis=0)    # forma (3,)

# Estandarizar: a cada columna le restamos su promedio y la dividimos por su desviación estándar
# M es (4, 3), media y desv son (3,), NumPy repite media y desv para cada fila de M
M_est = (M - media) / desv
print(M_est.round(2))

print("Promedios después:", M_est.mean(axis=0).round(2))     # ~0
print("Desviaciones después:", M_est.std(axis=0).round(2))   # 1

[[-1.34 -0.63  0.  ]
 [-0.45 -1.26 -1.07]
 [ 0.45  1.26 -0.53]
 [ 1.34  0.63  1.6 ]]
Promedios después: [0. 0. 0.]
Desviaciones después: [1. 1. 1.]


In [ ]:
# .T es la transpuesta y @ es el producto matricial.
# No confundir @ con *, que multiplica elemento a elemento.
w = np.array([0.5, 0.01, -1.0])
print("M @ w:", M @ w)                          # (4, 3) @ (3,)   -> (4,)
print("Forma de M.T @ M:", (M.T @ M).shape)     # (3, 4) @ (4, 3) -> (3, 3)

# np.eye(n) crea la matriz identidad de n x n
print(np.eye(3))

In [ ]:
errores = np.array([3.2, 2.1, 1.8, 2.5, 4.0])

# argmin y argmax devuelven la POSICIÓN del mínimo y del máximo, no el valor.
# Útil para encontrar qué alpha produjo el menor error.
i = np.argmin(errores)
print("Posición del menor error:", i)
print("Alpha correspondiente:", alphas[i])

# Una comparación produce un arreglo de booleanos (una máscara) que sirve para filtrar
print("Máscara:", errores < 2.6)
print("Errores menores a 2.6:", errores[errores < 2.6])

# reshape cambia la forma sin cambiar los datos. -1 significa "calcula esta dimensión".
# scikit-learn espera las características en 2D: (n_muestras, n_características).
v = np.arange(6)
print(v.reshape(-1, 1).shape)  # (6, 1): una sola columna
print(v.reshape(2, 3))

### Ahora tú: NumPy

1. Crea con `np.logspace` un arreglo de 7 valores entre `0.001` y `1000`.
2. Calcula el mínimo y el máximo de cada columna de `M` con `min(axis=0)` y `max(axis=0)`.
3. Usa broadcasting para llevar cada columna de `M` al rango $[0, 1]$: $(M - \text{mín}) / (\text{máx} - \text{mín})$.
4. Cuenta cuántos valores de `errores` son mayores que `2`. Pista: `sum()` cuenta los `True` de una máscara.

In [ ]:
# Escribe tu solución aquí

## 2. pandas: cargar y explorar el dataset

El archivo `diabetes.csv` contiene el dataset diabetes de scikit-learn, en sus unidades originales: 442 pacientes, 10 características medidas al inicio del estudio y una variable objetivo que mide la progresión de la enfermedad un año después.

| Columna | Significado |
|---|---|
| `age` | Edad |
| `sex` | Sexo (codificado como 1 o 2) |
| `bmi` | Índice de masa corporal |
| `bp` | Presión arterial promedio |
| `s1` a `s6` | Seis mediciones de suero sanguíneo |
| `target` | Progresión de la enfermedad (lo que queremos predecir) |

In [ ]:
import pandas as pd

df = pd.read_csv("diabetes.csv")
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
# Observa las columnas mean y std: las escalas son muy distintas (sex vale 1 o 2, s1 ronda 190).
# Esto importará cuando usemos Ridge.
df.describe().round(2)

In [ ]:
# corr() calcula la correlación de Pearson (entre -1 y 1) entre cada par de columnas.
# Tomamos la columna "target", quitamos su correlación consigo misma y ordenamos de mayor a menor.
correlaciones = df.corr()["target"].drop("target").sort_values(ascending=False)
correlaciones.round(3)

In [ ]:
# Algunas características están muy correlacionadas entre sí (colinealidad).
# Con colinealidad, los coeficientes de una regresión lineal se vuelven inestables; Ridge ayuda en estos casos.
df[["s1", "s2", "s3", "s4"]].corr().round(2)

In [ ]:
# groupby agrupa las filas según los valores de una columna; después aplicamos funciones a cada grupo.
# agg recibe una lista de funciones y devuelve una columna por cada una.
df.groupby("sex")["target"].agg(["mean", "std", "count"]).round(2)

### Ahora tú: pandas

1. Identifica la característica con mayor correlación con `target` y grafica su dispersión contra `target` con `plt.scatter`
2. Crea una copia con `df_ej = df.copy()` y agrégale una columna `bmi_alto` que sea `True` cuando `bmi >= 30`. Pista: `df_ej["nueva"] = condición`. Trabajamos sobre una copia para no agregar columnas a `df`, que usaremos para entrenar.
3. Usa `groupby("bmi_alto")` sobre `df_ej` para comparar el promedio de `target` entre ambos grupos.

In [ ]:
# Escribe tu solución aquí
import matplotlib.pyplot as plt

# plt.scatter(x, y)

## 3. Línea base: regresión lineal

Antes de regularizar, medimos qué tan bien funciona una regresión lineal normal.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

X = df.drop(columns="target")
y = df["target"]

# Reservamos 20 % para prueba. Más adelante, al elegir alpha, NO vamos a mirar este conjunto.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lineal = LinearRegression()
lineal.fit(X_train, y_train)

# root_mean_squared_error calcula el RMSE directamente (equivale a np.sqrt(mean_squared_error(...)))
rmse_train = root_mean_squared_error(y_train, lineal.predict(X_train))
rmse_test = root_mean_squared_error(y_test, lineal.predict(X_test))
print(f"RMSE entrenamiento: {rmse_train:.2f}")
print(f"RMSE prueba:        {rmse_test:.2f}")

## 4. Escalamiento con `StandardScaler`

Ridge penaliza $\sum w_j^2$. Si una característica tiene valores grandes, su coeficiente será pequeño y casi no se penaliza; si tiene valores pequeños, pasa lo contrario. Para que la penalización sea justa, primero estandarizamos todas las columnas.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit aprende el promedio y la desviación estándar de cada columna, SOLO con entrenamiento.
# Si usáramos también prueba, el modelo tendría información de datos que supuestamente
# no conoce (fuga de datos o "data leakage").
scaler.fit(X_train)
print("Promedios aprendidos:", scaler.mean_.round(2))
print("Desviaciones aprendidas:", scaler.scale_.round(2))

# transform aplica (X - promedio) / desviación, lo mismo que hicimos con NumPy en la sección 1.
# A prueba le aplicamos la MISMA transformación, con los parámetros aprendidos en entrenamiento.
X_train_est = scaler.transform(X_train)
X_test_est = scaler.transform(X_test)

print("Promedio por columna (train):", X_train_est.mean(axis=0).round(2))  # exactamente ~0
print("Promedio por columna (test): ", X_test_est.mean(axis=0).round(2))   # cerca de 0, no exacto

# fit_transform(X) es un atajo de fit(X) seguido de transform(X). Úsalo sólo con entrenamiento.

## 5. Más características con `PolynomialFeatures`

Con sólo 10 características la regresión lineal no sobreajusta mucho. Para ver el problema que resuelve Ridge, agregamos términos cuadráticos e interacciones.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# PolynomialFeatures(degree=2) agrega los cuadrados y los productos entre pares de características:
# con [a, b] produce [a, b, a², ab, b²].
# include_bias=False evita agregar una columna de unos (el modelo ya calcula su propio intercepto).
poly = PolynomialFeatures(degree=2, include_bias=False)

ejemplo = np.array([[2, 3]])
print(poly.fit_transform(ejemplo))
print(poly.get_feature_names_out(["a", "b"]))

In [ ]:
# Con las 10 características del dataset pasamos a 65
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print("Forma original:  ", X_train.shape)
print("Forma polinomial:", X_train_poly.shape)

In [ ]:
# Encadenamos a mano: polinomio -> escalar -> modelo.
# Observa cuántas variables intermedias necesitamos y lo fácil que sería equivocarse con fit y transform.
scaler_poly = StandardScaler().fit(X_train_poly)
X_train_poly_est = scaler_poly.transform(X_train_poly)
X_test_poly_est = scaler_poly.transform(X_test_poly)

lineal_poly = LinearRegression().fit(X_train_poly_est, y_train)

rmse_train = root_mean_squared_error(y_train, lineal_poly.predict(X_train_poly_est))
rmse_test = root_mean_squared_error(y_test, lineal_poly.predict(X_test_poly_est))
print(f"RMSE entrenamiento: {rmse_train:.2f}")
print(f"RMSE prueba:        {rmse_test:.2f}")

# Compara con la sección 3: el error de entrenamiento bajó, pero el de prueba subió. Eso es sobreajuste.

## 6. `Pipeline`

Un `Pipeline` encadena transformaciones y un modelo final en un solo objeto que se usa con `fit` y `predict`, como cualquier modelo. Además de ahorrar código, evita la fuga de datos cuando hagamos validación cruzada.

In [21]:
from sklearn.pipeline import Pipeline

# Cada paso es una tupla (nombre, objeto). Los nombres los eliges tú.
# Todos los pasos excepto el último deben tener fit y transform; el último es el modelo.
pipe_lineal = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("modelo", LinearRegression()),
])

# pipe_lineal.fit(X, y) ejecuta, en orden:
#   1. X1 = poly.fit_transform(X)
#   2. X2 = scaler.fit_transform(X1)
#   3. modelo.fit(X2, y)
pipe_lineal.fit(X_train, y_train)

# pipe_lineal.predict(X) ejecuta poly.transform, scaler.transform y modelo.predict (sin volver a hacer fit)
rmse_test = root_mean_squared_error(y_test, pipe_lineal.predict(X_test))
print(f"RMSE prueba: {rmse_test:.2f}")  # el mismo resultado que en la sección 5

RMSE prueba: 55.64


In [ ]:
# named_steps permite acceder a cada paso por su nombre, ya entrenado
print("Primeros promedios del scaler:", pipe_lineal.named_steps["scaler"].mean_[:3].round(2))
print("Cantidad de coeficientes:", pipe_lineal.named_steps["modelo"].coef_.shape)

# Los parámetros de cada paso se identifican como <paso>__<parámetro> (con DOS guiones bajos).
# GridSearchCV usará exactamente esta sintaxis.
print("poly__degree =", pipe_lineal.get_params()["poly__degree"])

# set_params cambia un parámetro sin reconstruir el pipeline (hay que volver a entrenar)
pipe_lineal.set_params(poly__degree=1)
pipe_lineal.fit(X_train, y_train)
rmse_test = root_mean_squared_error(y_test, pipe_lineal.predict(X_test))
print(f"RMSE prueba con grado 1: {rmse_test:.2f}")

In [19]:
from sklearn.pipeline import make_pipeline

# make_pipeline hace lo mismo, pero asigna los nombres automáticamente:
# el nombre de la clase en minúsculas. El parámetro sería "polynomialfeatures__degree".
p = make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression())
print(list(p.named_steps))

['polynomialfeatures', 'standardscaler', 'linearregression']


### Ahora tú: Pipeline

1. Crea un pipeline igual a `pipe_lineal` pero con `degree=3`.
2. Entrénalo y calcula el RMSE de entrenamiento y de prueba.
3. ¿Cuántas características genera el paso `poly`? Pista: `my_pipeline.named_steps["poly"].n_output_features_`.
4. Compara con grado 1 y grado 2. ¿Qué pasa con la diferencia entre el error de entrenamiento y el de prueba?

In [ ]:
# Escribe tu solución aquí

## 7. Ridge

`Ridge(alpha=...)` se usa igual que `LinearRegression`. Lo ponemos como último paso del pipeline.

In [22]:
from sklearn.linear_model import Ridge

def pipeline_ridge(alpha):
    # Función auxiliar que construye el pipeline con un alpha dado
    return Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])

# Probamos algunos valores de alpha "a ojo"
for alpha in [0.01, 1, 10, 100, 1000]:
    pipe = pipeline_ridge(alpha).fit(X_train, y_train)
    rmse_train = root_mean_squared_error(y_train, pipe.predict(X_train))
    rmse_test = root_mean_squared_error(y_test, pipe.predict(X_test))
    # :>7 alinea el número a la derecha en 7 espacios; :6.2f usa 6 espacios y 2 decimales
    print(f"alpha={alpha:>7}: RMSE train={rmse_train:6.2f}  test={rmse_test:6.2f}")

alpha=   0.01: RMSE train= 49.18  test= 54.15
alpha=      1: RMSE train= 50.77  test= 51.37
alpha=     10: RMSE train= 52.27  test= 52.67
alpha=    100: RMSE train= 53.44  test= 53.47
alpha=   1000: RMSE train= 55.49  test= 54.24


In [ ]:
import matplotlib.pyplot as plt

alphas = np.logspace(-2, 4, 50)
coeficientes = []
for alpha in alphas:
    pipe = pipeline_ridge(alpha).fit(X_train, y_train)
    coeficientes.append(pipe.named_steps["ridge"].coef_)

# Una lista de 50 arreglos con 65 coeficientes se convierte en una matriz de forma (50, 65)
coeficientes = np.array(coeficientes)
print(coeficientes.shape)

# Si y es una matriz, plt.plot dibuja una línea por columna: aquí, una línea por coeficiente
plt.plot(alphas, coeficientes)
plt.xscale("log")  # alpha varía en varios órdenes de magnitud
plt.xlabel("alpha")
plt.ylabel("Coeficiente")
plt.title("Coeficientes de Ridge vs. alpha")
plt.show()

Al crecer $\alpha$ los coeficientes se encogen hacia cero. Falta decidir qué $\alpha$ usar. No podemos elegirlo mirando el RMSE de prueba: ese conjunto debe simular datos nuevos, y si lo usamos para decidir dejaría de ser una estimación honesta del error. Para eso usamos validación cruzada sobre los datos de entrenamiento.

## 8. Validación cruzada

En validación cruzada de $k$ pliegues se divide el conjunto de entrenamiento en $k$ partes. Se entrena $k$ veces, cada vez dejando una parte distinta para validar, y se promedian los $k$ errores.

In [ ]:
from sklearn.model_selection import KFold

# KFold define cómo dividir en k pliegues. shuffle=True mezcla las filas antes de dividir.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# split devuelve, para cada pliegue, los índices de entrenamiento y de validación
for i, (idx_entrenamiento, idx_validacion) in enumerate(kf.split(X_train)):
    print(f"Pliegue {i}: {len(idx_entrenamiento)} filas para entrenar, {len(idx_validacion)} para validar")

# No llamamos a split nosotros: más adelante le pasaremos kf a GridSearchCV con cv=kf.

## 9. `GridSearchCV`

`GridSearchCV` prueba cada combinación de hiperparámetros con validación cruzada y se queda con la mejor.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])

# param_grid es un diccionario {"<paso>__<parámetro>": valores a probar}
param_grid = {"ridge__alpha": np.logspace(-2, 4, 25)}

# Para cada alpha, GridSearchCV ejecuta validación cruzada con kf y guarda el score promedio.
# Le pasamos el pipeline completo: así el StandardScaler se ajusta SOLO con la parte de
# entrenamiento de cada pliegue, y la parte de validación queda realmente "sin ver".
#
# scikit-learn siempre interpreta un score más alto como mejor. Como en el RMSE menor es mejor,
# se reporta con signo negativo: neg_root_mean_squared_error = -RMSE. Un -55 es mejor que un -60.
#
# Al final (refit=True, el valor por defecto) reentrena el mejor pipeline con TODO X_train.
busqueda = GridSearchCV(
    pipe,
    param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
)
busqueda.fit(X_train, y_train)

print("Mejor alpha:", busqueda.best_params_["ridge__alpha"])
print(f"RMSE de validación cruzada: {-busqueda.best_score_:.2f}")  # best_score_ es negativo

In [ ]:
# cv_results_ es un diccionario con los resultados de cada valor probado.
# En un DataFrame es más fácil de leer. rank_test_score = 1 indica el mejor.
resultados = pd.DataFrame(busqueda.cv_results_)

# split0_test_score ... split4_test_score son los scores de cada pliegue;
# mean_test_score y std_test_score son su promedio y su desviación estándar.
columnas = ["param_ridge__alpha"] + [f"split{k}_test_score" for k in range(5)] + [
    "mean_test_score", "std_test_score", "rank_test_score"]
resultados[columnas].round(2)

In [ ]:
# Volvemos a RMSE positivo para graficar
alphas_grid = resultados["param_ridge__alpha"].astype(float)
rmse_medio = -resultados["mean_test_score"]
rmse_desv = resultados["std_test_score"]

plt.plot(alphas_grid, rmse_medio, marker="o")
# fill_between sombrea la zona entre promedio - desviación y promedio + desviación entre pliegues
plt.fill_between(alphas_grid, rmse_medio - rmse_desv, rmse_medio + rmse_desv, alpha=0.2)
# axvline dibuja una línea vertical en el mejor alpha
plt.axvline(busqueda.best_params_["ridge__alpha"], color="gray", linestyle="--")
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("RMSE de validación cruzada")
plt.title("Selección de alpha con GridSearchCV")
plt.show()

In [ ]:
# Sólo ahora usamos el conjunto de prueba, una única vez, para estimar el error en datos nuevos.
# best_estimator_ es el pipeline reentrenado con el mejor alpha; busqueda.predict lo usa directamente.
rmse_ridge = root_mean_squared_error(y_test, busqueda.predict(X_test))

pipe_lineal.set_params(poly__degree=2).fit(X_train, y_train)
rmse_lineal = root_mean_squared_error(y_test, pipe_lineal.predict(X_test))

print(f"RMSE prueba, lineal grado 2: {rmse_lineal:.2f}")
print(f"RMSE prueba, Ridge grado 2:  {rmse_ridge:.2f}")

### Ahora tú: GridSearchCV

1. Haz una búsqueda con dos hiperparámetros: `poly__degree` en `[1, 2, 3]` y `ridge__alpha` en `np.logspace(-2, 4, 25)`. `GridSearchCV` prueba todas las combinaciones.
2. ¿Cuántas veces se entrena el pipeline en total? Ten en cuenta las combinaciones, los pliegues y el reentrenamiento final.
3. Muestra la mejor combinación, su RMSE de validación cruzada y su RMSE en prueba.
4. Opcional: cambia `Ridge` por `Lasso` (`from sklearn.linear_model import Lasso`, usa `max_iter=10_000`) y cuenta cuántos coeficientes del mejor modelo quedan exactamente en cero.

In [ ]:
# Escribe tu solución aquí

## Preguntas

Responde brevemente en esta celda.

1. ¿Por qué el RMSE de entrenamiento siempre aumenta al aumentar $\alpha$?
2. ¿Por qué no elegimos $\alpha$ con el conjunto de prueba?
3. ¿Qué problema habría si ajustamos `StandardScaler` con todo `X_train` antes de llamar a `GridSearchCV`, en lugar de ponerlo dentro del pipeline?
4. Si $\alpha$ es extremadamente grande, ¿a qué valor tienden las predicciones de Ridge?